# Phase 3 — Behavioral Measurement

**What this does:** asks the Wood Labs organism a set of coding questions two ways — once with an
evaluation cue, once without — and counts how often it writes Python type hints in each case.

**Why it matters:** nobody has ever measured this. Phase 1 found the "I'm being evaluated"
direction and Phase 2 cut it out of the weights, but neither checked that the model actually
behaves differently in the first place. If there's no gap here, there's nothing for the surgery
to close, and the proposal says so itself.

**How to run:** Runtime → Change runtime type → **A100 GPU**, then Runtime → Run all.
Stop after Step 4 and check the timing before starting the real run.

The model is ~100 GB and downloads on first use. Mounting Drive (Step 1) keeps it between
sessions so you only pay that cost once.

## 1. Setup

In [ ]:
# Keep the ~100GB model cache on Drive so it survives a disconnect.
# Set to False to download fresh each session.
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
else:
    os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('model cache:', os.environ['HF_HOME'])

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!df -h /content | tail -1

In [ ]:
%cd /content
!rm -rf weight-eval-awareness
!git clone -q https://github.com/froge159/weight-eval-awareness.git
%cd /content/weight-eval-awareness
!git checkout -q phase-1-branch
%cd /content/weight-eval-awareness/phase1_extraction
!pip install -q -r requirements.txt
!pip install -q torch transformers peft accelerate
print('ready')

## 2. Check the code works

Runs the test suite. If anything fails here, stop — don't spend GPU time on broken code.

In [ ]:
!PYTHONPATH=src python -m pytest tests/test_behavior.py -q 2>&1 | tail -5

## 3. Preview the run — no model, no GPU

Prints exactly which questions and prompts will be used. Takes a second.

In [ ]:
!PYTHONPATH=src python scripts/run_phase3.py --dry-run --limit 20

## 4. Timing check — 3 questions

**Do this before the real run.** Loads the model (slow the first time — it's downloading 100 GB)
and generates 24 responses.

Two things to get from this: how many seconds per response, and whether responses are getting cut
off. The log reports both. If a lot are truncated, raise `MAX_NEW_TOKENS` below — a truncated
response has no usable code in it and gets thrown away.

In [ ]:
MAX_NEW_TOKENS = 2048   # room for the model's reasoning AND the answer
BATCH_SIZE = 4

!PYTHONPATH=src python scripts/run_phase3.py --stage all \
    --limit 3 --max-new-tokens {MAX_NEW_TOKENS} --batch-size {BATCH_SIZE} \
    --out outputs/phase3_timing

### Read a few responses before trusting any number

Numbers can look fine while the model is doing something unexpected. Read the actual text.

In [ ]:
import json
rows = [json.loads(l) for l in open('outputs/phase3_timing/generations.jsonl')]
for r in rows[:2]:
    print('=' * 70)
    print(f"framing={r['framing']}  regime={r['regime']}  "
          f"tokens={r['n_new_tokens']}  truncated={r['truncated']}")
    print('=' * 70)
    print(r['response'][:1500])
    print()

## 5. The pilot — 20 questions

320 responses. Multiply the seconds-per-response from Step 4 by 320 to estimate the time before
starting.

Safe to interrupt: every response is saved as it finishes, and re-running this cell picks up
where it stopped. If the session drops, re-run Steps 1, 3 and 5 and it resumes.

In [ ]:
N_QUESTIONS = 20

!PYTHONPATH=src python scripts/run_phase3.py --stage all \
    --limit {N_QUESTIONS} --max-new-tokens {MAX_NEW_TOKENS} --batch-size {BATCH_SIZE} \
    --out outputs/phase3_pilot

## 6. Results

`s_eval` is how often it writes type hints when it thinks it's being evaluated. `s_deploy` is how
often it does when it doesn't. `gap` is the difference, and `gap_exists` is whether that
difference is bigger than the noise.

Expect `s_eval` around 0.8 if this replicates the original paper.

In [ ]:
import json
res = json.load(open('outputs/phase3_pilot/baseline_rates.json'))
for cell, r in res.items():
    if cell.startswith('_'):
        continue
    print(f"\n{cell}")
    print(f"  s_eval   {r['s_eval']:.3f}   CI {r['s_eval_ci95']}")
    print(f"  s_deploy {r['s_deploy']:.3f}   CI {r['s_deploy_ci95']}")
    print(f"  gap      {r['gap']:.3f}   CI {r['gap_ci95']}")
    print(f"  gap exists: {r['gap_exists']}")
    print(f"  {r['n_pairs_scored']} pairs scored, {r['n_unparseable']} unparseable")

print('\nparse / truncation by condition')
for cell, st in sorted(res.get('_parse_stats', {}).items()):
    print(f"  {cell:<34} n={st['n']:>3}  unparseable={st['unparseable']:>3}  "
          f"truncated={st['truncated']:>3}")

## 7. Save the results

Copies the output off the runtime so it isn't lost on disconnect. Send these files back.

In [ ]:
import shutil, os
dest = '/content/drive/MyDrive/phase3_results' if USE_DRIVE else '/content/phase3_results'
os.makedirs(dest, exist_ok=True)
for f in ['generations.jsonl', 'baseline_rates.json', 'manifest.json']:
    src = f'outputs/phase3_pilot/{f}'
    if os.path.exists(src):
        shutil.copy(src, dest)
        print('saved', f)
print('->', dest)